# Task 5 — Ordinal & one-hot encoding

**Problem-statement step:** *Apply ordinal and one-hot encoding based on the various types of categorical variables.*

* `Education` has a natural order → **ordinal** encoding.
* `Marital_Status` and `Country` are nominal → **one-hot** encoding.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

# Locate the raw data whether the notebook is opened from its own folder
# (notebooks/) or from the project root.
import os
_CANDIDATES = ["marketing_data.csv", "../marketing_data.csv",
               os.path.join(os.path.dirname(os.getcwd()), "marketing_data.csv")]
DATA_PATH = next((p for p in _CANDIDATES if os.path.exists(p)), "marketing_data.csv")
print("Using data file:", DATA_PATH)

Using data file: ../marketing_data.csv


In [2]:
REFERENCE_YEAR = 2015  # data compiled just after the last enrolment (Jun 2014)
SPEND_COLS = ["MntWines", "MntFruits", "MntMeatProducts",
              "MntFishProducts", "MntSweetProducts", "MntGoldProds"]
CHANNEL_COLS = ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]

def load_and_prepare(path=DATA_PATH, engineer=True):
    """Load -> fix dtypes -> clean categories -> impute income -> features."""
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()                     # ' Income ' -> 'Income'
    if not pd.api.types.is_numeric_dtype(df["Income"]):     # '$84,835.00' -> float
        df["Income"] = (df["Income"].astype("string")
                        .str.replace(r"[\$,]", "", regex=True).str.strip().astype(float))
    df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%m/%d/%y")
    df["Marital_Status"] = df["Marital_Status"].replace(
        {"Alone": "Single", "YOLO": "Single", "Absurd": "Single"})
    df["Income"] = df.groupby(["Education", "Marital_Status"])["Income"].transform(
        lambda s: s.fillna(s.median()))
    df["Income"] = df["Income"].fillna(df["Income"].median())
    if engineer:
        df["Kids"] = df["Kidhome"] + df["Teenhome"]
        df["Age"] = REFERENCE_YEAR - df["Year_Birth"]
        df["Total_Spending"] = df[SPEND_COLS].sum(axis=1)
        df["Total_Purchases"] = df[CHANNEL_COLS].sum(axis=1)
        df["Total_Accepted_Cmp"] = df[["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3",
                                       "AcceptedCmp4", "AcceptedCmp5"]].sum(axis=1)
        df["Has_Child"] = (df["Kids"] > 0).astype(int)
        df["Is_US"] = (df["Country"] == "US").astype(int)
    return df

def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)

def treat_outliers(df):
    """Drop impossible ages and winsorise heavy-tailed continuous columns."""
    df = df[df["Age"] <= 100].copy()
    for col in ["Income", "Total_Spending", "Total_Purchases", "NumWebVisitsMonth"]:
        lo, hi = iqr_bounds(df[col])
        df[col] = df[col].clip(lo, hi)
    return df

In [3]:
df = treat_outliers(load_and_prepare())
print("Shape:", df.shape)

Shape: (2237, 35)


## 1. Ordinal encoding of `Education`

In [4]:
EDU_ORDER = {"Basic": 0, "2n Cycle": 1, "Graduation": 2, "Master": 3, "PhD": 4}
df["Education_Ordinal"] = df["Education"].map(EDU_ORDER)
df[["Education", "Education_Ordinal"]].drop_duplicates().sort_values("Education_Ordinal")

,Education,Education_Ordinal
54,Basic,0
6,2n Cycle,1
0,Graduation,2
11,Master,3
5,PhD,4


## 2. One-hot encoding of `Marital_Status` and `Country`
`drop_first=True` avoids the dummy-variable trap.

In [5]:
df_enc = pd.get_dummies(df, columns=["Marital_Status", "Country"],
                        drop_first=True, dtype=int)
df_enc = df_enc.drop(columns=["Education"])  # keep the ordinal version
dummies = [c for c in df_enc.columns if c.startswith(("Marital_Status_", "Country_"))]
print(len(dummies), "dummy columns created:")
print(dummies)

11 dummy columns created:
['Marital_Status_Married', 'Marital_Status_Single', 'Marital_Status_Together', 'Marital_Status_Widow', 'Country_CA', 'Country_GER', 'Country_IND', 'Country_ME', 'Country_SA', 'Country_SP', 'Country_US']


In [6]:
df_enc[["Education_Ordinal"] + dummies[:4]].head(6)

,Education_Ordinal,Marital_Status_Married,Marital_Status_Single,Marital_Status_Together,Marital_Status_Widow
0,2,0,0,0,0
1,2,0,1,0,0
2,2,1,0,0,0
3,2,0,0,1,0
4,2,0,1,0,0
5,4,0,1,0,0


### Conclusion
The dataset is now fully numeric: `Education` collapses to a single ordinal column and the nominal fields expand into one-hot dummies — ready for correlation and modelling.